In [35]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"
TOC_DIR = BOUNDARIES_DIR / "tocs"
VILLAGES_DIR = BOUNDARIES_DIR / "villages"

PROJECT_CRS = "EPSG:2868"

jobs_toc_path = JOBS_DIR / "toc_jobs_2022.csv"
jobs_village_path = JOBS_DIR / "village_jobs_2022.csv"
toc_dem_path = TOC_DIR / "toc_demographics.gpkg"
villages_dem_path = VILLAGES_DIR / "village_demographics.gpkg"

jobs_toc_path, jobs_village_path, toc_dem_path, villages_dem_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/toc_jobs_2022.csv'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/village_jobs_2022.csv'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/tocs/toc_demographics.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/villages/village_demographics.gpkg'))

In [36]:
jobs_toc = pd.read_csv(jobs_toc_path)
jobs_village = pd.read_csv(jobs_village_path)

In [37]:
toc_dem = gpd.read_file(toc_dem_path)
villages_dem = gpd.read_file(villages_dem_path)

In [38]:
jobs_toc.columns

Index(['TOC_ID', 'OBJECTID', 'APPLICABIL', 'geometry', 'jobs_total',
       'acres_contributing', 'toc_area_sqft', 'toc_acres', 'jobs_per_acre'],
      dtype='object')

In [39]:
jobs_village.columns

Index(['Village_ID', 'OBJECTID', 'NAME', 'geometry', 'jobs_total',
       'acres_contributing', 'village_area_sqft', 'village_acres',
       'jobs_per_acre'],
      dtype='object')

In [40]:
toc_dem.columns

Index(['OBJECTID', 'APPLICABIL', 'TOC_ID', 'income_weighted', 'rent_weighted',
       'hh_total', 'median_income_approx', 'median_rent_approx',
       'intersect_acres', 'ASNQE001', 'ASS8E001', 'ASS8E002', 'ASS8E003',
       'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003', 'ASORE004', 'ASORE010',
       'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002', 'pct_drive_alone',
       'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh', 'pct_auto',
       'geometry'],
      dtype='object')

In [42]:
villages_dem.columns

Index(['OBJECTID', 'NAME', 'intersect_acres', 'hh_weighted', 'income_weighted',
       'rent_weighted', 'ASNQE001_w', 'ASS8E001_w', 'ASS8E002_w', 'ASS8E003_w',
       'ASS9E002_w', 'ASS9E003_w', 'ASORE001_w', 'ASORE003_w', 'ASORE004_w',
       'ASORE010_w', 'ASORE018_w', 'ASORE019_w', 'ASORE021_w', 'ASORE002_w',
       'median_income_approx', 'median_rent_approx', 'ASNQE001', 'ASS8E001',
       'ASS8E002', 'ASS8E003', 'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003',
       'ASORE004', 'ASORE010', 'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002',
       'pop_per_acre', 'hh_per_acre', 'pct_drive_alone', 'pct_transit',
       'pct_bike', 'pct_walk', 'pct_wfh', 'pct_auto', 'geometry'],
      dtype='object')

In [43]:
tocs_full = toc_dem.merge(
    jobs_toc[["TOC_ID", "APPLICABIL", "jobs_total"]],
    on=["TOC_ID", "APPLICABIL"],
    how="left",
    validate="1:1"
)

tocs_full[["TOC_ID", "APPLICABIL", "jobs_total", "ASS8E001", "ASS8E002"]].head()

,TOC_ID,APPLICABIL,jobs_total,ASS8E001,ASS8E002
0,0,TOD District - Gateway,4984.867123,4802.854853,4404.074570
1,1,TOD District - Eastlake Garfield,6847.046904,4021.409613,3379.818960
2,2,TOD District - Midtown,7099.922548,8259.602020,7079.314543
3,3,TOD District - Uptown,441.878230,5910.686353,5546.764459
4,4,TOD District - Solano,2006.609086,6067.435580,5518.487681


In [45]:
villages_full = villages_dem.merge(
    jobs_village[["OBJECTID", "NAME", "jobs_total"]],
    on=["OBJECTID", "NAME"],
    how="left",
    validate="1:1"
)

villages_full[["OBJECTID", "NAME", "jobs_total", "ASS8E001", "ASS8E002"]].head()

,OBJECTID,NAME,jobs_total,ASS8E001,ASS8E002
0,16,Ahwatukee Foothills,31854.214182,34930.417904,33008.638620
1,17,Laveen,38138.709412,20401.024379,19775.759721
2,18,South Mountain,60064.332489,43349.585529,41006.060284
3,19,Estrella,20731.840319,27735.695552,26650.951390
4,20,Central City,33900.231265,27345.355650,24096.557696


In [47]:
def safe_ratio(numerator, denominator):
    return np.where(
        (denominator > 0) & (~pd.isna(denominator)),
        numerator / denominator,
        np.nan
    )

# --- TOCs ---
tocs_full["jobs_per_housing_unit"] = safe_ratio(
    tocs_full["jobs_total"],
    tocs_full["ASS8E001"]  # total housing units
)

tocs_full["jobs_per_household"] = safe_ratio(
    tocs_full["jobs_total"],
    tocs_full["ASS8E002"]  # occupied housing units / households
)

# --- Villages ---
villages_full["jobs_per_housing_unit"] = safe_ratio(
    villages_full["jobs_total"],
    villages_full["ASS8E001"]
)

villages_full["jobs_per_household"] = safe_ratio(
    villages_full["jobs_total"],
    villages_full["ASS8E002"]
)

In [49]:
tocs_full[["APPLICABIL", "jobs_total", "jobs_per_housing_unit", "jobs_per_household"]].head(15)

,APPLICABIL,jobs_total,jobs_per_housing_unit,jobs_per_household
0,TOD District - Gateway,4984.867123,1.037897,1.131876
1,TOD District - Eastlake Garfield,6847.046904,1.702648,2.025862
2,TOD District - Midtown,7099.922548,0.859596,1.002911
3,TOD District - Uptown,441.878230,0.074759,0.079664
4,TOD District - Solano,2006.609086,0.330718,0.363616
5,TOD District - 19North,2635.120908,0.235368,0.252645
6,50th Street Station Area,1628.549532,1.070501,1.165644
7,Capitol Extension Area,3529.734836,1.487548,1.710546
8,I-10 West Extension Area,6465.078133,0.271578,0.283198
9,Northwest Extension Area II,1633.684459,0.280377,0.299333


In [50]:
villages_full[["NAME", "jobs_total", "jobs_per_housing_unit", "jobs_per_household"]].head(15)

,NAME,jobs_total,jobs_per_housing_unit,jobs_per_household
0,Ahwatukee Foothills,31854.214182,0.911933,0.965027
1,Laveen,38138.709412,1.869451,1.928558
2,South Mountain,60064.332489,1.385580,1.464767
3,Estrella,20731.840319,0.747479,0.777902
4,Central City,33900.231265,1.239707,1.406850
5,Camelback East,54700.185947,0.733486,0.814244
6,Maryvale,20996.374853,0.303084,0.315455
7,Encanto,18039.573404,0.598341,0.671793
8,Alhambra,24520.718073,0.463867,0.501048
9,North Mountain,18837.398628,0.256710,0.272139


In [51]:
village_full_path = VILLAGES_DIR / "village_jobs_full_2022.gpkg"
tocs_full_path = TOC_DIR / "toc_jobs_full_2022.gpkg"

villages_full.to_file(village_full_path, driver="GPKG", index=False)
tocs_full.to_file(tocs_full_path, driver="GPKG", index=False)